# From dental reports to useful data
## DENT 545 · Fall 2026 · Session 2 · HMM and named entity recognition

**Your mission:** A researcher has eight fictional oral-pathology reports. Identify reports with **currently affirmed epithelial dysplasia**, then list the finding, site and procedure for review.

By the end, you will annotate entity spans, train a small Hidden Markov Model, explain a predicted tag sequence, measure NER errors, and export a table. You will also see which parts of this task **NER alone cannot solve**.

**Time:** 70 minutes, in pairs. Basic Python is helpful; only small edits are needed. All data and tools are embedded. No package installation, GPU, API key or patient records are needed.

**Start:** In Google Colab choose **File → Save a copy in Drive**, then run each cell with ▶ or Shift+Enter. Do not run every cell at once: stop at each **Your turn** before revealing answers. A Google account is needed to run and save in Colab. The downloaded notebook also runs in a local Jupyter installation.

These hand-authored, simplified reports are educational examples, not clinical evidence. Use only fictional text in this exercise.

In [ ]:
#@title 0A. Load the teaching tools (run once; expand to inspect the HMM)
"""Small, transparent NER teaching tools. Python standard library only.

All examples in this project are fictional. This is not a clinical system.
"""
import csv
import html
import math
import re
from collections import Counter, defaultdict

LABELS = ("SITE", "PROCEDURE", "FINDING")
STATES = ("O",) + tuple(tag + "-" + label for label in LABELS for tag in ("B", "I"))
TOKEN = re.compile(r"\w+(?:[-']\w+)*|[^\w\s]", re.UNICODE)


def tokenize(text):
    return [{"text": m.group(), "start": m.start(), "end": m.end()} for m in TOKEN.finditer(text)]


def parse_report(report_id, marked):
    """Convert author markup into text and half-open character offsets."""
    chunks, entities, cursor = [], [], 0
    for match in re.finditer(r"\{([^{}]+)\}", marked):
        chunks.append(marked[cursor:match.start()])
        parts = match.group(1).split("|")
        label, surface = parts[0], parts[-1]
        assert label in LABELS and len(parts) in (2, 3)
        start = sum(map(len, chunks))
        entity = dict(start=start, end=start + len(surface), label=label, text=surface)
        if label == "FINDING":
            assert len(parts) == 3 and parts[1] in {"affirmed", "negated", "uncertain", "historical"}
            entity["assertion"] = parts[1]
        chunks.append(surface)
        entities.append(entity)
        cursor = match.end()
    chunks.append(marked[cursor:])
    report = dict(id=report_id, text="".join(chunks), entities=entities)
    bio_tags(report)  # Validate every boundary during authoring.
    return report


def bio_tags(report):
    tokens = tokenize(report["text"])
    tags = ["O"] * len(tokens)
    last_end = -1
    for entity in sorted(report["entities"], key=lambda e: e["start"]):
        assert entity["start"] >= last_end, "Overlapping spans are unsupported."
        assert report["text"][entity["start"]:entity["end"]] == entity["text"]
        indices = [i for i, token in enumerate(tokens)
                   if entity["start"] <= token["start"] and token["end"] <= entity["end"]]
        assert indices and tokens[indices[0]]["start"] == entity["start"]
        assert tokens[indices[-1]]["end"] == entity["end"]
        for j, i in enumerate(indices):
            tags[i] = ("B-" if j == 0 else "I-") + entity["label"]
        last_end = entity["end"]
    return tokens, tags


def allowed(previous, current):
    if current.startswith("I-"):
        return previous in ("B-" + current[2:], "I-" + current[2:])
    return True


def spans_from_tags(text, tokens, tags):
    assert len(tokens) == len(tags)
    entities, active, previous = [], None, "<START>"
    for token, tag in zip(tokens, tags):
        assert tag in STATES and allowed(previous, tag), "Invalid BIO sequence."
        if tag.startswith("B-") or tag == "O":
            if active:
                entities.append(active)
            active = None
        if tag.startswith("B-"):
            active = dict(start=token["start"], end=token["end"], label=tag[2:])
        elif tag.startswith("I-"):
            active["end"] = token["end"]
        previous = tag
    if active:
        entities.append(active)
    return [dict(e, text=text[e["start"]:e["end"]]) for e in entities]


class DictionaryNER:
    """Longest non-overlapping exact token sequence, case insensitive, TRAIN only."""
    def __init__(self, training):
        counts = defaultdict(Counter)
        for report in training:
            for entity in report["entities"]:
                key = tuple(t["text"].lower() for t in tokenize(entity["text"]))
                counts[key][entity["label"]] += 1
        self.lexicon = {key: labels.most_common(1)[0][0] for key, labels in counts.items()}

    def predict(self, text):
        tokens, result, i = tokenize(text), [], 0
        words = [t["text"].lower() for t in tokens]
        while i < len(tokens):
            candidates = [key for key in self.lexicon if tuple(words[i:i+len(key)]) == key]
            if not candidates:
                i += 1
                continue
            key = max(candidates, key=len)
            start, end = tokens[i]["start"], tokens[i+len(key)-1]["end"]
            result.append(dict(start=start, end=end, label=self.lexicon[key], text=text[start:end]))
            i += len(key)
        return result


class HMMNER:
    """Supervised first-order HMM with additive smoothing and log-space Viterbi.

    Word identities are lowercase; words absent from TRAIN map to <UNK>.
    BIO legality constrains and renormalizes the start/transition distributions.
    END is an explicit transition outcome; no token-level confidence is claimed.
    """
    def __init__(self, training, alpha=0.1):
        if alpha <= 0:
            raise ValueError("alpha must be positive")
        self.alpha = alpha
        self.start_counts, self.transitions, self.emissions = Counter(), defaultdict(Counter), defaultdict(Counter)
        self.vocab = {"<UNK>"}
        self.n_reports = len(training)
        if not training:
            raise ValueError("Provide at least one annotated training report")
        for report in training:
            tokens, tags = bio_tags(report)
            if not tokens:
                raise ValueError("Empty training reports are unsupported")
            self.start_counts[tags[0]] += 1
            for i, (token, tag) in enumerate(zip(tokens, tags)):
                word = token["text"].lower()
                self.vocab.add(word)
                self.emissions[tag][word] += 1
                if i:
                    self.transitions[tags[i-1]][tag] += 1
            self.transitions[tags[-1]]["<END>"] += 1
        self.emission_totals = {s: sum(self.emissions[s].values()) for s in STATES}
        self.transition_totals = {s: sum(self.transitions[s].values()) for s in STATES}

    def start_p(self, state):
        legal = [s for s in STATES if allowed("<START>", s)]
        return ((self.start_counts[state] + self.alpha) /
                (self.n_reports + self.alpha * len(legal))) if state in legal else 0.0

    def transition_p(self, previous, state):
        legal = [s for s in STATES if allowed(previous, s)] + ["<END>"]
        return ((self.transitions[previous][state] + self.alpha) /
                (self.transition_totals[previous] + self.alpha * len(legal))) if state in legal else 0.0

    def emission_p(self, state, word):
        word = word.lower() if word.lower() in self.vocab else "<UNK>"
        return ((self.emissions[state][word] + self.alpha) /
                (self.emission_totals[state] + self.alpha * len(self.vocab)))

    def decode(self, text):
        tokens = tokenize(text)
        if not tokens:
            return tokens, [], 0.0
        scores, back = [], []
        scores.append({s: (math.log(self.start_p(s)) + math.log(self.emission_p(s, tokens[0]["text"])))
                       if self.start_p(s) else -math.inf for s in STATES})
        back.append({})
        for token in tokens[1:]:
            current, pointers = {}, {}
            for state in STATES:
                options = [(scores[-1][prev] + math.log(self.transition_p(prev, state)), prev)
                           for prev in STATES if self.transition_p(prev, state)]
                best, predecessor = max(options, key=lambda pair: pair[0])
                current[state] = best + math.log(self.emission_p(state, token["text"]))
                pointers[state] = predecessor
            scores.append(current)
            back.append(pointers)
        best_score, final_state = max((scores[-1][s] + math.log(self.transition_p(s, "<END>")), s)
                                      for s in STATES)
        path = [final_state]
        for i in range(len(tokens)-1, 0, -1):
            path.append(back[i][path[-1]])
        return tokens, list(reversed(path)), best_score

    def predict(self, text):
        tokens, tags, _ = self.decode(text)
        return spans_from_tags(text, tokens, tags)

    def trace(self, text):
        tokens, tags, score = self.decode(text)
        rows = []
        for i, (token, tag) in enumerate(zip(tokens, tags)):
            previous = tags[i-1] if i else "<START>"
            rows.append(dict(token=token["text"], hidden_tag=tag, previous_tag=previous,
                             transition_or_start=round(self.transition_p(previous, tag) if i else self.start_p(tag), 5),
                             emission=round(self.emission_p(tag, token["text"]), 5),
                             unseen=token["text"].lower() not in self.vocab))
        return rows, score


def entity_key(entity):
    return entity["start"], entity["end"], entity["label"]


def evaluate(reports, predictor):
    tp = fp = fn = 0
    errors = []
    for report in reports:
        gold = {entity_key(e) for e in report["entities"]}
        predicted_entities = predictor(report["text"])
        predicted = {entity_key(e) for e in predicted_entities}
        tp += len(gold & predicted)
        fp += len(predicted - gold)
        fn += len(gold - predicted)
        for kind, keys in (("extra/wrong span", predicted-gold), ("missed gold span", gold-predicted)):
            for start, end, label in sorted(keys):
                errors.append(dict(report=report["id"], error=kind, text=report["text"][start:end], label=label))
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return dict(TP=tp, FP=fp, FN=fn, precision=round(precision, 3), recall=round(recall, 3), F1=round(f1, 3)), errors


def assertion_rule(text, entity):
    """Intentionally limited: preceding clause cues, then affirmed by default.

    It does NOT solve general negation, trailing cues, scope, or relation extraction.
    """
    before = re.split(r"[.;!?]", text[:entity["start"]].lower())[-1]
    if re.search(r"\b(possible|suspicious for|cannot exclude)\b", before):
        return "uncertain"
    if re.search(r"\b(history of|previous)\b", before):
        return "historical"
    if re.search(r"\b(no|without|negative for)\b", before):
        return "negated"
    return "affirmed"


def normalize_finding(surface):
    """An explicit task-specific mapping, separate from the learned NER model."""
    value = surface.lower()
    return "epithelial dysplasia" if "dysplasia" in value or value == "oed" else value


def structure(reports, predictor=None):
    """One row per finding mention. Sites/procedures are report-level lists.

    predictor=None uses gold spans AND gold assertions to establish the answer.
    With a predictor, assertions come from assertion_rule, never from gold labels.
    """
    rows = []
    for report in reports:
        entities = report["entities"] if predictor is None else predictor(report["text"])
        sites = "; ".join(e["text"] for e in entities if e["label"] == "SITE")
        procedures = "; ".join(e["text"] for e in entities if e["label"] == "PROCEDURE")
        for entity in entities:
            if entity["label"] == "FINDING":
                rows.append(dict(report_id=report["id"], finding=entity["text"],
                                 concept=normalize_finding(entity["text"]),
                                 assertion=entity["assertion"] if predictor is None else assertion_rule(report["text"], entity),
                                 sites_in_report=sites, procedures_in_report=procedures,
                                 start=entity["start"], end=entity["end"]))
    return rows


def cohort_ids(rows):
    return sorted({r["report_id"] for r in rows if r["concept"] == "epithelial dysplasia" and r["assertion"] == "affirmed"})


def table(rows):
    if not rows:
        print("No rows.")
        return
    try:
        from IPython.display import HTML, display
        columns = list(rows[0])
        markup = '<table style="border-collapse:collapse"><tr>' + ''.join('<th style="padding:6px;text-align:left">'+html.escape(str(c))+'</th>' for c in columns) + '</tr>'
        for row in rows:
            markup += '<tr>'+''.join('<td style="padding:6px;border-top:1px solid #ddd">'+html.escape(str(row[c]))+'</td>' for c in columns)+'</tr>'
        display(HTML(markup+'</table>'))
    except ImportError:
        for row in rows:
            print(row)


def highlight(text, entities):
    colors = dict(SITE="#d8edff", PROCEDURE="#eadfff", FINDING="#ffe2aa")
    chunks, cursor = [], 0
    for entity in sorted(entities, key=lambda e: e["start"]):
        chunks.append(html.escape(text[cursor:entity["start"]]))
        chunks.append('<mark style="background:'+colors[entity["label"]]+';padding:3px;border-radius:3px">'+html.escape(entity["text"])+' <small>['+entity["label"]+']</small></mark>')
        cursor = entity["end"]
    chunks.append(html.escape(text[cursor:]))
    markup = '<p style="line-height:2.4">'+''.join(chunks)+'</p>'
    try:
        from IPython.display import HTML, display
        display(HTML(markup))
    except ImportError:
        print(text)
        table(entities)


def export_csv(rows, path="ner_results.csv"):
    if not rows:
        raise ValueError("There are no finding rows to export")
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return path

In [ ]:
#@title 0B. Load the fictional reports (run once)
"""Hand-authored fictional oral-pathology snippets; no real patient data.

Markup: {SITE|surface}, {PROCEDURE|surface}, {FINDING|assertion|surface}.
Splits are fixed before fitting. Gold labels are educational conventions.
"""

MARKED = {
"train": [
"{PROCEDURE|Biopsy} of the {SITE|left lateral tongue} shows {FINDING|affirmed|mild epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|right buccal mucosa} shows {FINDING|affirmed|hyperkeratosis}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|floor of mouth} shows {FINDING|affirmed|moderate epithelial dysplasia}.",
"{PROCEDURE|Excision} of the {SITE|lower lip} shows {FINDING|affirmed|squamous cell carcinoma}.",
"{PROCEDURE|Excisional biopsy} of the {SITE|hard palate} shows {FINDING|affirmed|fibroma}.",
"{PROCEDURE|Biopsy} of the {SITE|ventral tongue} shows {FINDING|affirmed|severe epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|left gingiva} shows {FINDING|affirmed|chronic inflammation}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|soft palate} shows {FINDING|affirmed|epithelial dysplasia}.",
"{PROCEDURE|Excision} of the {SITE|right lateral tongue} shows {FINDING|affirmed|OED}.",
"{PROCEDURE|Biopsy} of the {SITE|left buccal mucosa} shows no {FINDING|negated|epithelial dysplasia}.",
"{PROCEDURE|Excision} of the {SITE|upper lip} shows no {FINDING|negated|carcinoma}.",
"{PROCEDURE|Biopsy} of the {SITE|right gingiva} shows possible {FINDING|uncertain|epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|floor of mouth} shows {FINDING|affirmed|hyperkeratosis} without {FINDING|negated|epithelial dysplasia}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|left lateral tongue}: {FINDING|affirmed|moderate epithelial dysplasia}.",
"{PROCEDURE|Biopsy} from the {SITE|hard palate}: {FINDING|affirmed|chronic inflammation}.",
"{PROCEDURE|Excision} from the {SITE|ventral tongue}: {FINDING|affirmed|mild epithelial dysplasia}.",
"{PROCEDURE|Excisional biopsy} from the {SITE|lower lip}: {FINDING|affirmed|fibroma}.",
"{PROCEDURE|Biopsy} from the {SITE|right buccal mucosa}: no {FINDING|negated|carcinoma}.",
"{PROCEDURE|Biopsy} from the {SITE|soft palate}: possible {FINDING|uncertain|squamous cell carcinoma}.",
"{PROCEDURE|Excision} of the {SITE|left gingiva}. History of {FINDING|historical|epithelial dysplasia}. Current finding: {FINDING|affirmed|hyperkeratosis}.",
"{PROCEDURE|Biopsy} of the {SITE|right lateral tongue}. Negative for {FINDING|negated|epithelial dysplasia}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|left buccal mucosa}. Cannot exclude {FINDING|uncertain|mild epithelial dysplasia}.",
"{PROCEDURE|Excision} of the {SITE|upper lip}. Diagnosis: {FINDING|affirmed|OED}.",
"{PROCEDURE|Biopsy} of the {SITE|right gingiva}. Diagnosis: {FINDING|affirmed|fibroma}.",
],
"dev": [
"{PROCEDURE|Excision} of the {SITE|right gingiva} shows {FINDING|affirmed|mild epithelial dysplasia}.",
"{PROCEDURE|Incisional biopsy} from the {SITE|lower lip}: no {FINDING|negated|epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|posterior tongue} shows {FINDING|affirmed|hyperkeratosis}.",
"{PROCEDURE|Excisional biopsy} of the {SITE|soft palate} shows possible {FINDING|uncertain|carcinoma}.",
"{PROCEDURE|Biopsy} of the {SITE|upper lip}. Diagnosis: {FINDING|affirmed|OED}.",
"{PROCEDURE|Biopsy} of the {SITE|left lateral tongue}. History of {FINDING|historical|carcinoma}. Current finding: {FINDING|affirmed|reactive atypia}.",
],
"test": [
"{PROCEDURE|Excisional biopsy} from the {SITE|floor of mouth}: {FINDING|affirmed|severe epithelial dysplasia}.",
"{PROCEDURE|Excision} of the {SITE|hard palate} shows no {FINDING|negated|epithelial dysplasia}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|right gingiva} shows {FINDING|affirmed|OED}.",
"{PROCEDURE|Biopsy} of the {SITE|lower lip} shows possible {FINDING|uncertain|moderate epithelial dysplasia}.",
"{PROCEDURE|Excision} from the {SITE|left buccal mucosa}: {FINDING|affirmed|squamous cell carcinoma}.",
"{PROCEDURE|Biopsy} of the {SITE|posterior buccal mucosa} shows {FINDING|affirmed|hyperkeratosis}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|ventral tongue}. Diagnosis: {FINDING|affirmed|mild epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|retromolar pad} shows {FINDING|affirmed|lichenoid mucositis}.",
],
"demo": [
"{PROCEDURE|Biopsy} from the {SITE|left lateral tongue} shows {FINDING|affirmed|mild epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|right buccal mucosa} shows no {FINDING|negated|epithelial dysplasia}.",
"{PROCEDURE|Excision} of the {SITE|floor of mouth} shows {FINDING|affirmed|OED}.",
"{PROCEDURE|Incisional biopsy} of the {SITE|lower lip} shows possible {FINDING|uncertain|epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|hard palate} shows {FINDING|affirmed|squamous cell carcinoma}.",
"{PROCEDURE|Biopsy} of the {SITE|left gingiva}. History of {FINDING|historical|epithelial dysplasia}. Current finding: {FINDING|affirmed|hyperkeratosis}.",
"{PROCEDURE|Excisional biopsy} of the {SITE|ventral tongue} shows {FINDING|affirmed|moderate epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|soft palate} shows {FINDING|affirmed|chronic inflammation} without {FINDING|negated|epithelial dysplasia}.",
],
"challenge": [
"{PROCEDURE|Biopsy} of the {SITE|lateral border of tongue} shows {FINDING|affirmed|oral epithelial dysplasia}.",
"{PROCEDURE|Biopsy} of the {SITE|lower lip}: {FINDING|negated|epithelial dysplasia} is not identified.",
"{PROCEDURE|Biopsy} of the {SITE|hard palate} shows not only {FINDING|affirmed|epithelial dysplasia} but also {FINDING|affirmed|chronic inflammation}.",
"{PROCEDURE|Biopsy} of the {SITE|left lateral tongue} shows {FINDING|affirmed|mild epithelial dysplasia}; {PROCEDURE|biopsy} of the {SITE|right gingiva} shows {FINDING|affirmed|hyperkeratosis}.",
"{PROCEDURE|Biopsy} of the {SITE|left buccal mucosa}: no {FINDING|negated|carcinoma}, but {FINDING|affirmed|epithelial dysplasia} is present.",
"{PROCEDURE|Punch biopsy} of the {SITE|upper labial mucosa} shows {FINDING|affirmed|verrucous hyperplasia}.",
]}

PREFIX = dict(train="T", dev="V", test="E", demo="D", challenge="C")
DATA = {split: [parse_report(f"{PREFIX[split]}{i:02}", marked) for i, marked in enumerate(reports, 1)]
        for split, reports in MARKED.items()}

print("Ready:", {name: len(reports) for name, reports in DATA.items()})

## 1 · Why do we need NER? (10 minutes)

Read the eight reports below. In this exercise **OED means oral epithelial dysplasia**. Include current, affirmed dysplasia of any grade. Exclude negated, possible and historical mentions. Count reports, not mentions.

**Your turn:** Before running the keyword search, put your selected report IDs into `my_report_ids`. Explain one inclusion and one exclusion to your partner. This is a report-review task, not a treatment decision.

In [ ]:
table([dict(report_id=r["id"], report=r["text"]) for r in DATA["demo"]])

In [ ]:
my_report_ids = []  # Example format: ["D01", "D03"]. Complete your own answer.
my_reason = ""     # Why is finding a word different from finding an affirmed condition?
print("Your selection:", my_report_ids)

In [ ]:
keyword_ids = [r["id"] for r in DATA["demo"] if "dysplasia" in r["text"].lower()]
print("Keyword search returns:", keyword_ids)
print("Number of reports:", len(keyword_ids))
print("Which false alarms or missed reports can you explain?")

<details><summary>Reveal after discussing: the answer and the reason</summary>

The correct report set is **D01, D03 and D07**. Keyword search returns six reports: D01, D02, D04, D06, D07 and D08. It finds two of the three target reports, includes four false alarms, and misses D03 because it uses OED.

NER finds spans such as **left lateral tongue → SITE**, **Biopsy → PROCEDURE**, and **mild epithelial dysplasia → FINDING**. A useful pipeline also needs **normalization** (OED → epithelial dysplasia) and **assertion detection** (affirmed, negated, uncertain, historical). A negated finding still has an entity span.

For several specimens in one report, linking each finding to its correct site requires additional relation extraction or review.
</details>

## 2 · Become the annotator (12 minutes)

Use these conventions consistently:

| Label | Include | Exclude |
|---|---|---|
| SITE | Full anatomical phrase, including laterality | `of the`, punctuation |
| PROCEDURE | Full procedure phrase, e.g. `Incisional biopsy` | Prepositions, punctuation |
| FINDING | Full finding, including severity when written | `no`, `possible`, `history of` |

We use **BIO**: `B-` begins an entity, `I-` continues the same entity, and `O` is outside. `O` is a tag, not the digit zero. A one-word entity has only `B-`. Whitespace and sentence punctuation are not included in entity spans.

**Your turn:** Complete `my_entities` using exact phrases copied from the report. One phrase is already provided. Then run the checker. This report contains one specimen; phrases are unique here.

In [ ]:
annotation_text = "Incisional biopsy of the left lateral tongue shows no mild epithelial dysplasia."
my_entities = [
    ("Incisional biopsy", "PROCEDURE"),
    # Add the SITE phrase and FINDING phrase as ("exact text", "LABEL").
]
print(annotation_text)

In [ ]:
annotation_gold = parse_report("A01", "{PROCEDURE|Incisional biopsy} of the {SITE|left lateral tongue} shows no {FINDING|negated|mild epithelial dysplasia}.")
student_spans = []
for phrase, label in my_entities:
    if label not in LABELS or not phrase or annotation_text.count(phrase) != 1:
        print("Check this entry:", (phrase, label), "— use one unique exact phrase and a listed label.")
        continue
    start = annotation_text.index(phrase)
    student_spans.append(dict(start=start, end=start+len(phrase), label=label, text=phrase))
annotation_score, annotation_errors = evaluate([annotation_gold], lambda _: student_spans)
table([annotation_score])
print("Perfect span-and-label agreement?", annotation_score["F1"] == 1.0)
SHOW_ANNOTATION_ANSWER = False  # Change to True only after your attempt.
if SHOW_ANNOTATION_ANSWER:
    highlight(annotation_text, annotation_gold["entities"])
    tokens, gold_tags = bio_tags(annotation_gold)
    table([dict(token=t["text"], BIO_tag=tag) for t, tag in zip(tokens, gold_tags)])

**Discuss:** Is `no` a FINDING? Should a negated entity be omitted? Why is `I-FINDING` invalid immediately after `O`? What would happen to annotation agreement if one person included `mild` and another did not?

Exact matching requires both the correct boundaries and the correct entity label. A partial span gives one false positive and one false negative.

## 3 · Teach a dictionary and an HMM (12 minutes)

The **dictionary baseline** memorizes entity phrases in the 24 training reports. It takes the longest matching token sequence. The **HMM** learns how BIO labels follow each other and how likely words are under each label.

| HMM ingredient | Here | Link to the lecture |
|---|---|---|
| Observations | Words and punctuation tokens | Observed moods in the weather example |
| Hidden states | BIO entity tags | Hidden weather states |
| Initial probability | Probability of the first tag | Initial weather distribution |
| Transition probability | Probability of a tag given the previous tag | Weather-to-weather probability |
| Emission probability | Probability of a word given its tag | Mood given weather |
| Decoding | Best complete BIO sequence for this text | Viterbi best weather sequence |

During supervised training, the hidden labels are supplied by annotators. At prediction time we infer them. This model estimates probabilities with counts and additive smoothing, maps unseen words to an unknown-word symbol, and forbids invalid BIO transitions. It uses first-order label context, not a language model's understanding of a report.

**Your turn:** Predict which system will do better on a site phrase it has never seen. Can either system guarantee correct boundaries?

In [ ]:
baseline = DictionaryNER(DATA["train"])
hmm = HMMNER(DATA["train"], alpha=0.1)
dev_comparison = []
for name, model in [("Dictionary", baseline), ("HMM", hmm)]:
    metrics, errors = evaluate(DATA["dev"], model.predict)
    dev_comparison.append(dict(model=name, **metrics))
table(dev_comparison)

These are **development** results: use them to diagnose errors and choose changes. The training, development and test reports are separate. Entity vocabulary intentionally overlaps, as it does in many tasks, but whole reports are not duplicated. The tiny, repetitive synthetic set is not evidence of real clinical performance.

**Read the numbers:** Precision = TP/(TP+FP); recall = TP/(TP+FN); F1 = their harmonic mean. TP requires exact character boundaries **and** entity type. Counts are pooled across reports (micro averaging). We do not use token accuracy because many tokens are simply `O`.

In [ ]:
dev_report = DATA["dev"][2]
print(dev_report["id"], dev_report["text"])
print("Dictionary prediction:")
highlight(dev_report["text"], baseline.predict(dev_report["text"]))
print("HMM prediction:")
highlight(dev_report["text"], hmm.predict(dev_report["text"]))
print("Human reference:")
highlight(dev_report["text"], dev_report["entities"])

## 4 · Follow the hidden tags (10 minutes)

For observations x and tags y, the model maximizes:

`P(y1) × P(x1|y1) × ∏[P(yt|y(t−1)) × P(xt|yt)] × P(END|yT)`

Viterbi keeps the best partial score for each possible ending tag and saves backpointers to recover the best complete sequence. The code sums log probabilities to avoid tiny floating-point products.

The table below shows the **selected path**, with its transition and emission probabilities. These are components of the joint score; they are **not** confidence percentages for each prediction. The whole dynamic program is in the expandable setup cell and `lab/ner_lab.py`.

**Your turn:** Locate the first `B-SITE`, its `I-SITE` continuation, and any unseen word. Explain why choosing a tag from only the current word can produce a different answer from choosing the best whole path.

In [ ]:
trace_text = dev_report["text"]
trace_rows, joint_log_score = hmm.trace(trace_text)
table(trace_rows)
print("Best joint log score, including END:", round(joint_log_score, 4))
print("A larger joint score compares paths for this same text; it is not a clinical certainty score.")

## 5 · Make one improvement, then freeze it (10 minutes)

Inspect development errors. Try adding **one newly written fictional training sentence** that addresses an observed error (for example, a site boundary). Use the annotation syntax below. Do not copy a development/test report verbatim into training, and do not inspect test errors until you freeze your choice.

`{PROCEDURE|Biopsy} of the {SITE|your fictional site phrase} shows {FINDING|affirmed|hyperkeratosis}.`

**Your turn:** Write your prediction before running the model. If scores do not improve, explain why; improvement is not guaranteed. Unknown words are deliberately handled very simply here. Additional data, richer features, or a different model could help.

In [ ]:
_, dev_errors = evaluate(DATA["dev"], hmm.predict)
table(dev_errors)

In [ ]:
my_change_prediction = ""  # Which error should your added example address?
EXTRA_MARKED = []          # Add one annotated fictional sentence as a quoted string.
extra_training = [parse_report(f"NEW{i}", text) for i, text in enumerate(EXTRA_MARKED, 1)]
existing_texts = {r["text"] for split in DATA.values() for r in split}
if any(r["text"] in existing_texts for r in extra_training):
    raise ValueError("Write a new sentence; do not copy an existing report into training.")
final_hmm = HMMNER(DATA["train"] + extra_training, alpha=0.1)
final_dev_metrics, _ = evaluate(DATA["dev"], final_hmm.predict)
table([dict(model="Your HMM on development data", **final_dev_metrics)])

**Freeze your change now.** Run the held-out test once for the final comparison. If you subsequently change the model based on these test results, this set has become development data and you would need a new test set for an unbiased evaluation.

In [ ]:
test_results = []
for name, model in [("Dictionary", baseline), ("Original HMM", hmm), ("Your HMM", final_hmm)]:
    metrics, _ = evaluate(DATA["test"], model.predict)
    test_results.append(dict(model=name, **metrics))
table(test_results)
_, test_errors = evaluate(DATA["test"], final_hmm.predict)
table(test_errors)

## 6 · Turn spans into a table (10 minutes)

Return to the eight opening reports. We now add two **separate, deliberately limited** steps:

1. Normalize OED and dysplasia phrases to `epithelial dysplasia` using an explicit mapping.
2. Inspect preceding text in the same clause for uncertainty, history and negation cues; otherwise default to affirmed.

These rules were written for teaching, not learned by the HMM. They can fail on trailing negation and complex scope. We retain mention offsets and the original report so a person can check the output.

One row represents **one finding mention**. `sites_in_report` and `procedures_in_report` list co-occurring entities; they do not assert which site/procedure belongs to a finding. That distinction matters when a report has more than one specimen.

In [ ]:
predicted_rows = structure(DATA["demo"], final_hmm.predict)
table(predicted_rows)
predicted_cohort = cohort_ids(predicted_rows)
gold_cohort = cohort_ids(structure(DATA["demo"]))
print("Pipeline-selected reports:", predicted_cohort)
print("Human-reference reports:  ", gold_cohort)
print("False inclusions:", sorted(set(predicted_cohort)-set(gold_cohort)))
print("Missed reports:", sorted(set(gold_cohort)-set(predicted_cohort)))
print("Keyword-only reports:     ", keyword_ids)
output_path = export_csv(predicted_rows)
print("Created", output_path, "— open the Colab Files panel to download it.")

## 7 · Break the pipeline (6 minutes / extension)

Try these challenge cases **after** final evaluation. Their purpose is error analysis, not estimating population performance. Use the human entity spans first to isolate assertion mistakes; then compare the HMM predictions if you have time.

**Your turn:** Find a report where perfect NER is still insufficient. Explain why the two-specimen report cannot safely produce finding–site pairs from a list of all sites.

In [ ]:
challenge_assertions = []
for report in DATA["challenge"]:
    print(report["id"], report["text"])
    for entity in report["entities"]:
        if entity["label"] == "FINDING":
            rule_result = assertion_rule(report["text"], entity)
            challenge_assertions.append(dict(report=report["id"], finding=entity["text"],
                                              gold=entity["assertion"], rule=rule_result,
                                              correct=rule_result == entity["assertion"]))
table(challenge_assertions)

## Exit ticket (submit your copied notebook)

1. In two sentences, explain why keyword search returned the wrong report set. Include one report ID.
2. Show your completed annotation and explain one BIO boundary choice.
3. Identify observations, hidden states, transitions and emissions in this HMM.
4. Report exact-span precision, recall and F1 for both baseline systems. Describe your change and one error that remains.
5. Explain one failure that belongs to assertion detection or relation extraction rather than NER. Attach `ner_results.csv` and state what would require human review.

**Before leaving:** Save your notebook copy. Do not upload real patient information.

### Resources

- [Course repository](https://github.com/Tahereh-Firoozi/DENT-545-Fall-2026)
- [Student worksheet](https://github.com/Tahereh-Firoozi/DENT-545-Fall-2026/blob/main/student_worksheet.md)
- Jurafsky & Martin, [Sequence labeling for parts of speech and named entities](https://web.stanford.edu/~jurafsky/slp3/old_aug24/17.pdf), for further reading on BIO and HMMs.
- [Colab FAQ](https://research.google.com/colaboratory/faq.html), for notebook access and saving.

The instructor guide contains solutions and is publicly accessible; attempt the activities first.